# 比亚迪港股 (01211.HK) — 技术指标计算 Notebook

本 Notebook 演示四大经典技术指标的手动计算过程：

| 指标 | 中文名 | 用途 |
|:----|:------|:----|
| **RSI** | 相对强弱指标 | 判断超买/超卖 (14日) |
| **MACD** | 指数平滑异同平均线 | 趋势强度与方向 (12/26/9) |
| **Bollinger Bands** | 布林带 | 波动率与价格位置 (20, 2σ) |
| **ATR** | 平均真实波幅 | 市场波动性度量 (14日) |

> 数据：比亚迪港股 (01211.HK) 过去一年日线，来源 Tushare Pro

---
## 0. 环境准备

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['PingFang SC', 'Heiti TC', 'Microsoft YaHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

print('✅ 环境就绪')

---
## 1. 数据获取

> 从 Tushare Pro 获取比亚迪港股 (01211.HK) 日线数据。
> 港股数据返回的货币单位为 **港元 (HKD)**。

In [ ]:
# ========== 数据源配置 ==========
USE_CACHED = True   # True=读本地CSV, False=从API拉取
CSV_PATH = 'data/raw/01211.HK_20250703_20260703.csv'
# ===============================

if USE_CACHED and os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    print(f'✅ 从本地 CSV 读取: {len(df)} 条')
else:
    print('⚠️ 尝试从 Tushare MCP 拉取...')
    # 此处使用 Tushare MCP 接口或 Python SDK
    # 实际使用时可在下方填入 token 并取消注释
    # import tushare as ts
    # pro = ts.pro_api('your_token')
    # df = pro.hk_daily(ts_code='01211.HK', start_date='20250703', end_date='20260703')
    raise FileNotFoundError(f'请先确保 {CSV_PATH} 存在，或将 USE_CACHED 改为 False 并配置 Token')

df['trade_date'] = pd.to_datetime(df['trade_date'])
df.sort_values('trade_date', inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'日期范围: {df["trade_date"].min().date()} ~ {df["trade_date"].max().date()}')
print(f'最新收盘价: HK${df["close"].iloc[-1]:.2f}')
df[['trade_date', 'open', 'high', 'low', 'close', 'vol']].head()

---
## 2. RSI — 相对强弱指标 (14日)

### 计算公式
1. 计算每日价格变动: `delta = close - close.shift(1)`
2. 分离上涨 (gain) 和下跌 (loss)
3. 用指数移动平均 (EMA) 平滑 gain 和 loss
4. `RS = avg_gain / avg_loss`
5. `RSI = 100 - 100 / (1 + RS)`

### 判断标准
- `RSI > 70`: 超买 (可能回调)
- `RSI < 30`: 超卖 (可能反弹)
- `RSI = 50`: 多空平衡

In [ ]:
period = 14

# Step 1: 每日价格变动
delta = df['close'].diff()

# Step 2: 分离涨跌
gain = delta.where(delta > 0, 0.0)
loss = (-delta).where(delta < 0, 0.0)

# Step 3 & 4: 指数平滑平均
avg_gain = gain.ewm(alpha=1/period, min_periods=period).mean()
avg_loss = loss.ewm(alpha=1/period, min_periods=period).mean()

# Step 5 & 6: 计算 RSI
rs = avg_gain / avg_loss
df['rsi'] = 100 - (100 / (1 + rs))

# 检查最近的值
latest = df[['trade_date', 'close', 'rsi']].tail(5)
latest['rsi'] = latest['rsi'].round(1)
print('最近 5 个交易日的 RSI：')
print(latest.to_string(index=False))
print(f'\n📊 当前 RSI = {df["rsi"].iloc[-1]:.1f}')
if df['rsi'].iloc[-1] > 70:
    print('⚠️ 超买区域 (>70)')
elif df['rsi'].iloc[-1] < 30:
    print('⚠️ 超卖区域 (<30)')
else:
    print('✅ 正常区间')

---
## 3. MACD — 指数平滑异同移动平均线

### 计算公式
1. `EMA_fast = close.ewm(span=12).mean()`
2. `EMA_slow = close.ewm(span=26).mean()`
3. `DIF = EMA_fast - EMA_slow`
4. `DEA = DIF.ewm(span=9).mean()`
5. `MACD 柱 = DIF - DEA`

### 判断标准
- **金叉**: DIF 上穿 DEA → 买入信号
- **死叉**: DIF 下穿 DEA → 卖出信号
- **柱体 > 0**: 多头动能增强

In [ ]:
fast, slow, signal = 12, 26, 9

# Step 1-2: 计算快慢 EMA
ema_fast = df['close'].ewm(span=fast, min_periods=fast).mean()
ema_slow = df['close'].ewm(span=slow, min_periods=slow).mean()

# Step 3: DIF
df['macd_dif'] = ema_fast - ema_slow

# Step 4: DEA (信号线)
df['macd_dea'] = df['macd_dif'].ewm(span=signal, min_periods=signal).mean()

# Step 5: MACD 柱
df['macd_hist'] = df['macd_dif'] - df['macd_dea']

# 判断最新信号
latest_dif = df['macd_dif'].iloc[-1]
latest_dea = df['macd_dea'].iloc[-1]
latest_hist = df['macd_hist'].iloc[-1]

print(f'最新 MACD 值：')
print(f'  DIF (快线):  {latest_dif:.3f}')
print(f'  DEA (信号线): {latest_dea:.3f}')
print(f'  MACD 柱:    {latest_hist:.3f}')

if latest_dif > latest_dea:
    print(f'  DIF 在 DEA 上方 → 多头优势')
else:
    print(f'  DIF 在 DEA 下方 → 空头优势')

# 金叉/死叉检测
prev_dif = df['macd_dif'].iloc[-2]
prev_dea = df['macd_dea'].iloc[-2]
if prev_dif < prev_dea and latest_dif > latest_dea:
    print('🔴 金叉信号！DIF 上穿 DEA')
elif prev_dif > prev_dea and latest_dif < latest_dea:
    print('🟢 死叉信号！DIF 下穿 DEA')

---
## 4. 布林带 (20日, 2 倍标准差)

### 计算公式
1. `中轨 = close.rolling(20).mean()`
2. `标准差 = close.rolling(20).std()`
3. `上轨 = 中轨 + 2 × 标准差`
4. `下轨 = 中轨 - 2 × 标准差`
5. `带宽 = (上轨 - 下轨) / 中轨 × 100%`
6. `%B = (收盘价 - 下轨) / (上轨 - 下轨)`

### 判断标准
- 价格接触/突破上轨 → 超买或强趋势
- 价格接触/跌破下轨 → 超卖或强趋势
- 带宽收窄 (Squeeze) → 即将突破
- `%B > 1`: 价格在上轨之上
- `%B < 0`: 价格在下轨之下

In [ ]:
bb_period, bb_std = 20, 2

# Step 1-2: 计算中轨和标准差
df['boll_mid'] = df['close'].rolling(bb_period).mean()
bb_std_val = df['close'].rolling(bb_period).std()

# Step 3-4: 上下轨
df['boll_upper'] = df['boll_mid'] + bb_std * bb_std_val
df['boll_lower'] = df['boll_mid'] - bb_std * bb_std_val

# Step 5: 带宽
df['boll_bandwidth'] = (df['boll_upper'] - df['boll_lower']) / df['boll_mid'] * 100

# Step 6: %B
df['boll_pctb'] = (df['close'] - df['boll_lower']) / (df['boll_upper'] - df['boll_lower'])

# 最新值
last = df.iloc[-1]
print(f'最新布林带值：')
print(f'  上轨: HK${last["boll_upper"]:.2f}')
print(f'  中轨: HK${last["boll_mid"]:.2f}')
print(f'  下轨: HK${last["boll_lower"]:.2f}')
print(f'  收盘价: HK${last["close"]:.2f}')
print(f'  带宽: {last["boll_bandwidth"]:.2f}%')
print(f'  %B:  {last["boll_pctb"]:.3f}')

if last['close'] > last['boll_upper']:
    print('⚠️ 价格突破上轨！')
elif last['close'] < last['boll_lower']:
    print('⚠️ 价格跌破下轨！')
else:
    print('✅ 价格在布林带内部运行')

---
## 5. ATR — 平均真实波幅 (14日)

### 计算公式
1. `TR1 = high - low` (当日振幅)
2. `TR2 = |high - pre_close|` (最高到昨收)
3. `TR3 = |low - pre_close|` (最低到昨收)
4. `TR = max(TR1, TR2, TR3)` (真实波幅)
5. 首日 `ATR = TR.rolling(14).mean()`
6. 之后 `ATR = (前日 ATR × 13 + 当日 TR) / 14`

> ATR 越大 → 波动越大，适合缩小仓位、扩大止损

In [ ]:
atr_period = 14

# Step 1-4: 计算 True Range
high_low = df['high'] - df['low']
high_pc = (df['high'] - df['pre_close']).abs()
low_pc = (df['low'] - df['pre_close']).abs()

df['true_range'] = pd.concat([high_low, high_pc, low_pc], axis=1).max(axis=1)

# Step 5-6: 计算 ATR (Wilder's Smoothed ATR)
df['atr'] = df['true_range'].rolling(atr_period).mean()

# 精度检查
last_tr = df['true_range'].iloc[-1]
last_atr = df['atr'].iloc[-1]

print(f'最新 ATR 值：')
print(f'  真实波幅 (TR):   HK${last_tr:.3f}')
print(f'  平均真实波幅 (ATR): HK${last_atr:.3f}')
print(f'  当前收盘价:        HK${df["close"].iloc[-1]:.2f}')
print(f'  ATR / 收盘价:      {last_atr / df["close"].iloc[-1] * 100:.2f}%')

# 常用止损参考
entry_price = df['close'].iloc[-1]
print(f'\n📐 基于 ATR 的止损参考（2 倍 ATR）：')
print(f'  多头止损: HK${entry_price - 2 * last_atr:.2f}')
print(f'  空头止损: HK${entry_price + 2 * last_atr:.2f}')

---
## 6. 综合可视化

将四个指标绘制在同一张图上，便于综合判断。

In [ ]:
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(5, 1, height_ratios=[3, 1.2, 1.5, 1.2, 1], hspace=0.08)

dates = df['trade_date']

# ─── Panel 1: 价格 + 布林带 ───
ax1 = fig.add_subplot(gs[0])
ax1.plot(dates, df['close'], color='#e74c3c', linewidth=1.2, label='收盘价')
ax1.plot(dates, df['boll_upper'], color='#5b8ff9', linewidth=0.7, alpha=0.5, label='上轨')
ax1.plot(dates, df['boll_mid'], color='#f39c12', linewidth=0.8, alpha=0.6, label='中轨')
ax1.plot(dates, df['boll_lower'], color='#5b8ff9', linewidth=0.7, alpha=0.5, label='下轨')
ax1.fill_between(dates, df['boll_upper'], df['boll_lower'], alpha=0.08, color='#5b8ff9')
ax1.set_ylabel('价格 (HK$)', fontsize=10)
ax1.set_title('比亚迪 (01211.HK) — 技术指标综合面板', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left', fontsize=8, ncol=4)
ax1.grid(True, alpha=0.2)

# ─── Panel 2: 成交量 ───
ax2 = fig.add_subplot(gs[1], sharex=ax1)
colors = ['#e74c3c' if df['pct_chg'].iloc[i] >= 0 else '#27ae60' for i in range(len(df))]
ax2.bar(dates, df['vol'] / 10000, color=colors, alpha=0.5, width=0.8)
ax2.set_ylabel('成交量 (万股)', fontsize=10)
ax2.grid(True, alpha=0.2)

# ─── Panel 3: MACD ───
ax3 = fig.add_subplot(gs[2], sharex=ax1)
ax3.plot(dates, df['macd_dif'], color='#e74c3c', linewidth=1, label='DIF')
ax3.plot(dates, df['macd_dea'], color='#5b8ff9', linewidth=1, label='DEA')
hist_colors = ['#e74c3c' if v >= 0 else '#27ae60' for v in df['macd_hist']]
ax3.bar(dates, df['macd_hist'], color=hist_colors, alpha=0.4, width=0.8)
ax3.axhline(y=0, color='#999', linewidth=0.5)
ax3.set_ylabel('MACD', fontsize=10)
ax3.legend(loc='upper left', fontsize=8, ncol=3)
ax3.grid(True, alpha=0.2)

# ─── Panel 4: RSI ───
ax4 = fig.add_subplot(gs[3], sharex=ax1)
ax4.plot(dates, df['rsi'], color='#8e44ad', linewidth=1, label='RSI(14)')
ax4.axhline(y=70, color='#e74c3c', linestyle='--', linewidth=0.7, alpha=0.5)
ax4.axhline(y=30, color='#27ae60', linestyle='--', linewidth=0.7, alpha=0.5)
ax4.fill_between(dates, 70, 100, alpha=0.08, color='#e74c3c')
ax4.fill_between(dates, 0, 30, alpha=0.08, color='#27ae60')
ax4.set_ylabel('RSI', fontsize=10)
ax4.set_ylim(0, 100)
ax4.legend(loc='upper left', fontsize=8)
ax4.grid(True, alpha=0.2)

# ─── Panel 5: ATR ───
ax5 = fig.add_subplot(gs[4], sharex=ax1)
ax5.fill_between(dates, df['atr'], alpha=0.3, color='#f39c12')
ax5.plot(dates, df['atr'], color='#e67e22', linewidth=1, label='ATR(14)')
ax5.set_ylabel('ATR (HK$)', fontsize=10)
ax5.set_xlabel('交易日期', fontsize=10)
ax5.legend(loc='upper left', fontsize=8)
ax5.grid(True, alpha=0.2)

for ax in [ax1, ax2, ax3, ax4]:
    plt.setp(ax.xaxis.get_majorticklabels(), visible=False)

ax5.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax5.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
fig.autofmt_xdate()
plt.show()
plt.close('all')

---
## 7. 综合信号汇总

整合四个指标的当前信号，方便快速判断。

In [ ]:
last = df.iloc[-1]

# RSI 信号
rsi_signal = '超买' if last['rsi'] > 70 else ('超卖' if last['rsi'] < 30 else '中性')

# MACD 信号
if last['macd_dif'] > last['macd_dea']:
    macd_trend = '多头'
else:
    macd_trend = '空头'

# 布林带信号
if last['close'] > last['boll_upper']:
    boll_signal = '突破上轨'
elif last['close'] < last['boll_lower']:
    boll_signal = '跌破下轨'
else:
    boll_signal = '带内运行'

# ATR 波动率
atr_ratio = last['atr'] / last['close'] * 100
volatility = '高波动' if atr_ratio > 5 else ('低波动' if atr_ratio < 2 else '中等波动')

print(f'{"="*42}')
print(f'  比亚迪 (01211.HK) — 技术指标信号汇总')
print(f'{"="*42}')
print(f'  日期:          {last["trade_date"].date()}')
print(f'  收盘价:        HK${last["close"]:.2f}')
print(f'{"─"*42}')
print(f'  RSI({14:>2d}):        {last["rsi"]:.1f}  → {rsi_signal}')
print(f'  MACD DIF:      {last["macd_dif"]:.3f}  → {macd_trend}')
print(f'  MACD DEA:      {last["macd_dea"]:.3f}')
print(f'  布林带:        {boll_signal}')
print(f'  带宽:          {last["boll_bandwidth"]:.2f}%')
print(f'  ATR:           HK${last["atr"]:.3f} ({volatility})')
print(f'  ATR/收盘价:    {atr_ratio:.2f}%')
print(f'{"─"*42}')
print(f'  💡 2倍ATR多头止损: HK${last["close"] - 2 * last["atr"]:.2f}')
print(f'{"="*42}')
print('* 以上信号仅供参考，不构成投资建议 *')

---
**数据来源**: [Tushare Pro](https://tushare.pro)  |  **指标规范**: `indicators_spec.yaml`

**声明**: 本 Notebook 仅演示技术指标的计算方法，不构成任何投资建议。